# Motor simbólico paso a paso para teorías \(f(R)\)

Este notebook reproduce la lógica de la deducción del Beamer **sin tomar la ecuación final de \(f(R)\) como punto de partida**.

La secuencia es:

$
\mathcal L_\xi L \text{ por dos rutas}
\;\Longrightarrow\;
P^{ab}=-2\mathcal R^{ab}
\;\Longrightarrow\;
\delta(\sqrt{-g}L)
\;\Longrightarrow\;
P^{abcd}\delta R_{abcd}
$

$
\Longrightarrow\;
\text{identidad de Palatini}
\;\Longrightarrow\;
\text{dos integraciones por partes}
\;\Longrightarrow\;
E_{ab}
\;\Longrightarrow\;
\text{especialización al input }L=f(R).
$

La filosofía es deliberadamente **derivación-first**:

- **SymPy** calcula todo lo que depende realmente del input: $f_R$, $f_{RR}$, $f_{RRR}$, coeficientes y simplificaciones.
- Las operaciones de geometría tensorial abstracta —simetrías de Riemann, compatibilidad métrica, identidad de Palatini e integración por partes covariante— se implementan como reglas explícitas y se muestran una por una.
- La ecuación de campo final se ensambla **solo después** de haber construido por separado $\mathcal R_{ab}$, el término métrico y $-2\nabla^m\nabla^nP_{amnb}$.

> **Convención:** se usa la misma normalización del Beamer: en presencia de materia, $E_{ab}=\tfrac12 T_{ab}$.

## Correspondencia con el Beamer

| Bloque del Beamer | Parte del notebook |
|---|---|
| Derivada de Lie e identidad principal | Etapa 1 |
| Cálculo de $P^{abcd}$ y $\mathcal R_{ab}$ | Etapas 2 y 3 |
| Variación de $\sqrt{-g}L$ | Etapa 4 |
| Descomposición de $P^{abcd}\delta R_{abcd}$ | Etapa 5 |
| Palatini + dos integraciones por partes | Etapa 6 |
| Tensor general $E_{ab}$ | Fin de la etapa 6 |
| Evaluación de $-2\nabla\nabla P$ | Etapa 7 |
| Condición $\nabla_aP^{abcd}=0$ y orden | Etapa 9 |

El notebook separa con cuidado la **deducción general** de la **especialización $L=f(R)$**. Así, en versiones futuras se podrá reutilizar la misma arquitectura para otros lagrangianos.

In [5]:
import sympy as sp
from dataclasses import dataclass
from IPython.display import display, Math, Markdown

sp.init_printing()

# ------------------------------------------------------------------
# Símbolos escalares
# ------------------------------------------------------------------
R = sp.Symbol("R", real=True)
alpha, beta, gamma, Lambda, mu = sp.symbols(
    "alpha beta gamma Lambda mu", real=True
)

# ------------------------------------------------------------------
# Utilidades de visualización
# ------------------------------------------------------------------
def tex(expr):
    # LaTeX compacto para expresiones escalares de SymPy.
    return sp.latex(sp.factor(sp.simplify(expr)))

def section(title):
    display(Markdown(f"### {title}"))

def note(text):
    display(Markdown(text))

def show_eq(lhs, rhs, boxed=False):
    body = rf"{lhs} = {rhs}"
    if boxed:
        body = rf"\boxed{{{body}}}"
    display(Math(body))

def show_chain(*lines):
    body = r"\begin{aligned}" + r"\\".join(lines) + r"\end{aligned}"
    display(Math(body))

# ------------------------------------------------------------------
# Modelo f(R)
# ------------------------------------------------------------------
@dataclass(frozen=True)
class FRModel:
    L: sp.Expr

    def __post_init__(self):
        object.__setattr__(self, "L", sp.sympify(self.L))

    @property
    def f_R(self):
        return sp.simplify(sp.diff(self.L, R))

    @property
    def f_RR(self):
        return sp.simplify(sp.diff(self.L, R, 2))

    @property
    def f_RRR(self):
        return sp.simplify(sp.diff(self.L, R, 3))

    @property
    def P_prefactor(self):
        # P^{abcd} = P_prefactor * (g^{ac}g^{bd}-g^{ad}g^{bc})
        return sp.simplify(self.f_R / 2)

    def validate(self):
        # Todo símbolo libre distinto de R se interpreta como parámetro.
        if self.L.has(sp.Derivative):
            raise ValueError(
                "El input debe ser una función algebraica de R, sin derivadas explícitas."
            )
        return True

    def summary(self):
        self.validate()
        section("Input y derivadas escalares calculadas por SymPy")
        show_eq("L", tex(self.L))
        show_eq(r"f_R \equiv \dfrac{df}{dR}", tex(self.f_R))
        show_eq(r"f_{RR} \equiv \dfrac{d^2f}{dR^2}", tex(self.f_RR))
        show_eq(r"f_{RRR} \equiv \dfrac{d^3f}{dR^3}", tex(self.f_RRR))

## Input del usuario

Edita **solo** la línea `L_input = ...` y ejecuta el notebook desde esta celda hacia abajo.

Ejemplos válidos:

```python
L_input = R
L_input = R - 2*Lambda
L_input = R + alpha*R**2
L_input = R + alpha*R**2 + beta*R**3
L_input = sp.exp(beta*R)
L_input = R + alpha/R
```

In [20]:
# ================================================================
# EDITAR SOLO ESTA LÍNEA
# ================================================================
#L_input = R + alpha*R**2 - 2*Lambda
L_input = R

model = FRModel(L_input)
model.summary()

### Input y derivadas escalares calculadas por SymPy

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Etapa 1. Reproducir la identidad $P^{ab}=-2\mathcal R^{ab}$

Esta parte sigue el primer bloque conceptual del Beamer. Todavía **no** usamos ninguna ecuación final de $f(R)$.

In [21]:
def derive_lie_identity():
    section(r"1A. Primer cálculo de la derivada de Lie: usar que \(L\) es escalar")

    show_eq(r"\mathcal L_\xi L", r"\xi^m\nabla_mL")
    show_eq(
        r"\nabla_mL",
        r"P^{ab}\nabla_mg_{ab}+P^{ijkl}\nabla_mR_{ijkl}",
    )
    note(
        r"Con la conexión de Levi--Civita, \(\nabla_mg_{ab}=0\). "
        r"Por tanto, la primera ruta da:"
    )
    show_eq(
        r"\mathcal L_\xi L",
        r"\xi^mP^{ijkl}\nabla_mR_{ijkl}",
        boxed=True,
    )

    section(r"1B. Segundo cálculo: variar los argumentos de \(L\)")

    show_eq(
        r"\delta L",
        r"P^{ab}\delta g_{ab}+P^{ijkl}\delta R_{ijkl}",
    )
    show_eq(r"\delta_\xi g_{ab}", r"\mathcal L_\xi g_{ab}")
    show_eq(r"\delta_\xi R_{ijkl}", r"\mathcal L_\xi R_{ijkl}")
    show_eq(
        r"\mathcal L_\xi L",
        r"P^{ab}\mathcal L_\xi g_{ab}"
        r"+P^{ijkl}\mathcal L_\xi R_{ijkl}",
    )

    section("1C. Simplificar el término métrico")

    show_eq(
        r"\mathcal L_\xi g_{ab}",
        r"\nabla_a\xi_b+\nabla_b\xi_a",
    )
    show_chain(
        r"P^{ab}\mathcal L_\xi g_{ab}"
        r"&=P^{ab}\nabla_a\xi_b+P^{ab}\nabla_b\xi_a",
        r"&=P^{ab}\nabla_a\xi_b+P^{ba}\nabla_a\xi_b",
        r"&=2P^{ab}\nabla_a\xi_b",
    )
    show_eq(
        r"P^{ab}\mathcal L_\xi g_{ab}",
        r"2P^{ab}\nabla_a\xi_b",
        boxed=True,
    )

    section("1D. Simplificar el término de curvatura")

    note(
        r"Se codifican exactamente las simetrías usadas en el Beamer: "
        r"\[P^{ijkl}=-P^{jikl}=-P^{ijlk},\qquad P^{ijkl}=P^{klij}.\]"
    )
    show_eq(
        r"\mathcal L_\xi R_{ijkl}",
        r"\xi^m\nabla_mR_{ijkl}"
        r"+R_{mjkl}\nabla_i\xi^m"
        r"+R_{imkl}\nabla_j\xi^m"
        r"+R_{ijml}\nabla_k\xi^m"
        r"+R_{ijkm}\nabla_l\xi^m",
    )

    # El CAS escalar no manipula índices abstractos. La regla tensorial
    # aplicada aquí es la misma reducción canónica del Beamer.
    copies_after_canonicalization = [1, 1, 1, 1]
    multiplicity = sum(copies_after_canonicalization)

    note(
        r"Tras renombrar índices mudos y usar las simetrías de \(P\) y \(R\), "
        f"los cuatro términos se reducen a {multiplicity} copias del mismo término."
    )
    show_eq(
        r"P^{ijkl}\mathcal L_\xi R_{ijkl}",
        r"P^{ijkl}\xi^m\nabla_mR_{ijkl}"
        r"+4P^{ijkl}R_{mjkl}\nabla_i\xi^m",
    )

    section(r"1E. Definir \(\mathcal R^{ab}\) y comparar ambas rutas")

    show_eq(r"\mathcal R^{ab}", r"P^{aijk}R^b{}_{ijk}", boxed=True)
    show_eq(
        r"\mathcal L_\xi L",
        r"P^{ijkl}\xi^m\nabla_mR_{ijkl}"
        r"+2\left(P^{ab}+2\mathcal R^{ab}\right)\nabla_a\xi_b",
    )
    note(
        r"La primera ruta no contiene un término independiente proporcional a "
        r"\(\nabla_a\xi_b\). Como \(\xi^a\) es arbitrario, su coeficiente debe anularse."
    )
    show_eq(r"P^{ab}+2\mathcal R^{ab}", r"0")
    show_eq(r"P^{ab}", r"-2\mathcal R^{ab}", boxed=True)

derive_lie_identity()

### 1A. Primer cálculo de la derivada de Lie: usar que \(L\) es escalar

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Con la conexión de Levi--Civita, \(\nabla_mg_{ab}=0\). Por tanto, la primera ruta da:

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

La primera ruta no contiene un término independiente proporcional a \(\nabla_a\xi_b\). Como \(\xi^a\) es arbitrario, su coeficiente debe anularse.

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Etapa 2. Calcular $P^{abcd}$ desde el input $L=f(R)$

Ahora sí entra el lagrangiano elegido por el usuario. El notebook no inserta una forma prefabricada de la ecuación de campo: empieza por calcular

$f_R=\frac{df}{dR}$

y luego usa la regla de la cadena para construir $P^{abcd}$.

In [22]:
def derive_P(model):
    section("2A. Derivada escalar del lagrangiano")
    show_eq(r"f_R", tex(model.f_R))

    section(r"2B. Escribir \(R\) con la antisimetría correcta de Riemann")
    show_eq(
        "R",
        r"\frac12"
        r"\left(g^{ac}g^{bd}-g^{ad}g^{bc}\right)R_{abcd}",
    )

    section("2C. Aplicar la regla de la cadena")
    show_chain(
        r"P^{abcd}"
        r"&\equiv\left(\frac{\partial L}{\partial R_{abcd}}\right)_{g}",
        r"&=\frac{d f}{dR}"
        r"\frac{\partial R}{\partial R_{abcd}}",
        rf"&=\left({tex(model.f_R)}\right)"
        r"\frac12\left(g^{ac}g^{bd}-g^{ad}g^{bc}\right)",
    )

    show_eq(
        r"P^{abcd}",
        rf"{tex(model.P_prefactor)}"
        r"\left(g^{ac}g^{bd}-g^{ad}g^{bc}\right)",
        boxed=True,
    )

    note(
        r"El factor \(f_R\) es un escalar. Por eso, las simetrías algebraicas "
        r"de \(P^{abcd}\) provienen enteramente del antisimetizador métrico."
    )

derive_P(model)

### 2A. Derivada escalar del lagrangiano

<IPython.core.display.Math object>

### 2B. Escribir \(R\) con la antisimetría correcta de Riemann

<IPython.core.display.Math object>

### 2C. Aplicar la regla de la cadena

<IPython.core.display.Math object>

<IPython.core.display.Math object>

El factor \(f_R\) es un escalar. Por eso, las simetrías algebraicas de \(P^{abcd}\) provienen enteramente del antisimetizador métrico.

# Etapa 3. Calcular \(\mathcal R_{ab}\) desde el \(P^{abcd}\) obtenido

El objetivo es reproducir la misma contracción hecha en el caso Einstein--Hilbert, pero ahora con el factor \(f_R\) generado por el input.

In [23]:
def derive_Rcal(model):
    section(r"3A. Bajar el primer índice de \(P^{mijk}\)")

    show_eq(
        r"P^{mijk}",
        rf"\frac12\left({tex(model.f_R)}\right)"
        r"\left(g^{mj}g^{ik}-g^{mk}g^{ij}\right)",
    )
    show_eq(
        r"P_a{}^{ijk}",
        rf"\frac12\left({tex(model.f_R)}\right)"
        r"\left(\delta_a^j g^{ik}-\delta_a^k g^{ij}\right)",
    )

    section(r"3B. Contraer con \(R_{bijk}\)")

    show_chain(
        r"\mathcal R_{ab}"
        r"&=P_a{}^{ijk}R_{bijk}",
        rf"&=\frac12\left({tex(model.f_R)}\right)"
        r"\left(g^{ik}R_{biak}-g^{ij}R_{bija}\right)",
        rf"&=\frac12\left({tex(model.f_R)}\right)"
        r"\left(R_{ab}-(-R_{ab})\right)",
        rf"&=\left({tex(model.f_R)}\right)R_{{ab}}",
    )
    show_eq(
        r"\mathcal R_{ab}",
        rf"\left({tex(model.f_R)}\right)R_{{ab}}",
        boxed=True,
    )

    section("3C. Usar la identidad obtenida con la derivada de Lie")

    show_eq(
        r"\left(\frac{\partial L}{\partial g_{ab}}\right)_{R_{ijkl}}",
        rf"-2\left({tex(model.f_R)}\right)R^{{ab}}",
    )
    show_eq(
        r"\left(\frac{\partial L}{\partial g^{ab}}\right)_{R_{ijkl}}",
        rf"2\left({tex(model.f_R)}\right)R_{{ab}}",
        boxed=True,
    )

derive_Rcal(model)

### 3A. Bajar el primer índice de \(P^{mijk}\)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3B. Contraer con \(R_{bijk}\)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 3C. Usar la identidad obtenida con la derivada de Lie

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Etapa 4. Variar \(\sqrt{-g}L\)

Todavía no tocamos el término de curvatura. Primero se separa la variación del determinante y la variación del lagrangiano exactamente como en el Beamer.

In [24]:
def derive_density_variation(model):
    section("4A. Variación del determinante")
    show_eq(
        r"\delta\sqrt{-g}",
        r"-\frac12\sqrt{-g}\,g_{ab}\delta g^{ab}",
    )

    section("4B. Variación del lagrangiano")
    show_eq(
        r"\delta L",
        rf"2\left({tex(model.f_R)}\right)R_{{ab}}\delta g^{{ab}}"
        r"+P^{abcd}\delta R_{abcd}",
    )

    section("4C. Combinar ambos aportes")
    show_eq(
        r"\delta(\sqrt{-g}L)",
        r"\sqrt{-g}\left["
        rf"\left(2\left({tex(model.f_R)}\right)R_{{ab}}"
        rf"-\frac12g_{{ab}}\left({tex(model.L)}\right)\right)\delta g^{{ab}}"
        r"+P^{abcd}\delta R_{abcd}\right]",
        boxed=True,
    )

derive_density_variation(model)

### 4A. Variación del determinante

<IPython.core.display.Math object>

### 4B. Variación del lagrangiano

<IPython.core.display.Math object>

### 4C. Combinar ambos aportes

<IPython.core.display.Math object>

# Etapa 5. Descomponer \(P^{abcd}\delta R_{abcd}\)

Aquí aparece el primer ajuste importante del coeficiente métrico. No se salta directamente al tensor \(E_{ab}\).

In [25]:
def derive_riemann_split(model):
    section("5A. Variar el Riemann completamente covariante")

    show_eq(
        r"\delta R_{abcd}",
        r"\delta g_{ae}R^e{}_{bcd}+g_{ae}\delta R^e{}_{bcd}",
    )
    show_eq(
        r"P^{abcd}\delta R_{abcd}",
        r"P^{abcd}R^e{}_{bcd}\delta g_{ae}"
        r"+P^{abcd}g_{ae}\delta R^e{}_{bcd}",
    )

    section(r"5B. Reescribir el primer aporte con \(\delta g^{ab}\)")

    show_eq(r"\delta g_{ae}", r"-g_{am}g_{en}\delta g^{mn}")
    show_eq(
        r"P^{abcd}R^e{}_{bcd}\delta g_{ae}",
        r"-\mathcal R_{mn}\delta g^{mn}",
    )
    show_eq(
        r"P^{abcd}R^e{}_{bcd}\delta g_{ae}",
        rf"-\left({tex(model.f_R)}\right)R_{{mn}}\delta g^{{mn}}",
    )

    section("5C. Resultado de la separación")
    show_eq(
        r"P^{abcd}\delta R_{abcd}",
        rf"-\left({tex(model.f_R)}\right)R_{{ab}}\delta g^{{ab}}"
        r"+P^{abcd}g_{ae}\delta R^e{}_{bcd}",
        boxed=True,
    )

    note(
        r"Al insertarlo en la etapa anterior, el coeficiente \(2\mathcal R_{ab}\) "
        r"se reduce a \(\mathcal R_{ab}\). El resto se obtiene con Palatini."
    )

derive_riemann_split(model)

### 5A. Variar el Riemann completamente covariante

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5B. Reescribir el primer aporte con \(\delta g^{ab}\)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 5C. Resultado de la separación

<IPython.core.display.Math object>

Al insertarlo en la etapa anterior, el coeficiente \(2\mathcal R_{ab}\) se reduce a \(\mathcal R_{ab}\). El resto se obtiene con Palatini.

# Etapa 6. Identidad de Palatini y dos integraciones por partes

Esta etapa es universal para cualquier \(L(g,R_{abcd})\) sin derivadas explícitas del Riemann. El \(P^{abcd}\) que entra aquí es el que ya fue calculado desde el input.

In [26]:
def derive_palatini_and_ibp():
    section("6A. Identidad de Palatini")

    show_eq(
        r"\delta R^e{}_{bcd}",
        r"\nabla_c\delta\Gamma^e{}_{db}"
        r"-\nabla_d\delta\Gamma^e{}_{cb}",
    )
    show_eq(
        r"P^{abcd}g_{ae}\delta R^e{}_{bcd}",
        r"2P^{abcd}g_{ae}\nabla_c\delta\Gamma^e{}_{db}",
    )

    section(r"6B. Sustituir \(\delta\Gamma\)")

    show_eq(
        r"\delta\Gamma^e{}_{db}",
        r"\frac12g^{ei}"
        r"\left(\nabla_d\delta g_{bi}"
        r"+\nabla_b\delta g_{di}"
        r"-\nabla_i\delta g_{db}\right)",
    )
    note(
        r"Usando \(\nabla g=0\), las simetrías de \(P^{abcd}\), "
        r"\(\delta g_{ab}=\delta g_{ba}\) y renombrado de índices mudos:"
    )
    show_eq(
        r"P^{abcd}g_{ae}\delta R^e{}_{bcd}",
        r"2P^{ibjd}\nabla_j\nabla_b\delta g_{di}",
        boxed=True,
    )

    section("6C. Primera integración por partes covariante")

    show_eq(
        r"2P^{ibjd}\nabla_j\nabla_b\delta g_{di}",
        r"\nabla_j\left(2P^{ibjd}\nabla_b\delta g_{di}\right)"
        r"-2(\nabla_jP^{ibjd})\nabla_b\delta g_{di}",
    )

    section("6D. Segunda integración por partes covariante")

    show_eq(
        r"-2(\nabla_cP^{ijcd})\nabla_j\delta g_{di}",
        r"-\nabla_j\left(2\delta g_{di}\nabla_cP^{ijcd}\right)"
        r"+2\delta g_{di}\nabla_j\nabla_cP^{ijcd}",
    )

    section("6E. Identificar el término de borde")

    show_eq(
        r"\delta v^j",
        r"2P^{ibjd}\nabla_b\delta g_{di}"
        r"-2\delta g_{di}\nabla_cP^{ijcd}",
        boxed=True,
    )

    show_eq(
        r"P^{abcd}g_{ae}\delta R^e{}_{bcd}",
        r"-2\nabla^m\nabla^nP_{amnb}\,\delta g^{ab}"
        r"+\nabla_j\delta v^j",
    )

    section(r"6F. Volver a \(P^{abcd}\delta R_{abcd}\)")

    show_eq(
        r"P^{abcd}\delta R_{abcd}",
        r"-\mathcal R_{ab}\delta g^{ab}"
        r"-2\nabla^m\nabla^nP_{amnb}\delta g^{ab}"
        r"+\nabla_j\delta v^j",
        boxed=True,
    )

    section(r"6G. Tensor de campo general antes de especializar \(P\)")

    show_eq(
        r"E_{ab}",
        r"\mathcal R_{ab}"
        r"-\frac12g_{ab}L"
        r"-2\nabla^m\nabla^nP_{amnb}",
        boxed=True,
    )

derive_palatini_and_ibp()

### 6A. Identidad de Palatini

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 6B. Sustituir \(\delta\Gamma\)

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

### 6F. Volver a \(P^{abcd}\delta R_{abcd}\)

<IPython.core.display.Math object>

### 6G. Tensor de campo general antes de especializar \(P\)

<IPython.core.display.Math object>

# Etapa 7. Calcular \(-2\nabla^m\nabla^nP_{amnb}\) desde el \(P\) obtenido

Esta es la etapa que evita usar la fórmula final de \(f(R)\) como un dato conocido.

In [27]:
def derivative_term_expanded_tex(model):
    # Expansión por regla de la cadena de
    # g_ab Box(f_R)-nabla_a nabla_b(f_R).
    f2 = sp.simplify(model.f_RR)
    f3 = sp.simplify(model.f_RRR)

    pieces = []
    if f2 != 0:
        pieces.append(
            rf"\left({tex(f2)}\right)"
            r"\left(g_{ab}\Box R-\nabla_a\nabla_bR\right)"
        )
    if f3 != 0:
        pieces.append(
            rf"\left({tex(f3)}\right)"
            r"\left[g_{ab}(\nabla R)^2"
            r"-(\nabla_aR)(\nabla_bR)\right]"
        )
    return " + ".join(pieces) if pieces else "0"


def derive_double_divergence(model):
    section(r"7A. Bajar los índices de \(P^{abcd}\) con el patrón requerido")

    show_eq(
        r"P_{amnb}",
        rf"\frac12\left({tex(model.f_R)}\right)"
        r"\left(g_{an}g_{mb}-g_{ab}g_{mn}\right)",
        boxed=True,
    )

    section("7B. Aplicar dos derivadas covariantes")

    show_chain(
        r"\nabla^m\nabla^nP_{amnb}"
        rf"&=\frac12\nabla^m\nabla^n"
        rf"\left[\left({tex(model.f_R)}\right)"
        r"\left(g_{an}g_{mb}-g_{ab}g_{mn}\right)\right]",
        rf"&=\frac12\left["
        r"g_{an}g_{mb}\nabla^m\nabla^n"
        rf"\left({tex(model.f_R)}\right)"
        r"-g_{ab}g_{mn}\nabla^m\nabla^n"
        rf"\left({tex(model.f_R)}\right)\right]",
    )

    note(
        r"En la segunda línea se usó únicamente la compatibilidad métrica "
        r"\(\nabla_\lambda g_{\mu\nu}=0\)."
    )

    section("7C. Contraer las métricas")

    show_eq(
        r"\nabla^m\nabla^nP_{amnb}",
        rf"\frac12\left["
        rf"\nabla_a\nabla_b\left({tex(model.f_R)}\right)"
        rf"-g_{{ab}}\Box\left({tex(model.f_R)}\right)"
        r"\right]",
    )

    section(r"7D. Multiplicar por el factor \(-2\) que venía de la variación")

    show_eq(
        r"-2\nabla^m\nabla^nP_{amnb}",
        rf"g_{{ab}}\Box\left({tex(model.f_R)}\right)"
        rf"-\nabla_a\nabla_b\left({tex(model.f_R)}\right)",
        boxed=True,
    )

    section("7E. Expandir la regla de la cadena covariante")

    show_eq(
        r"\nabla_a\nabla_b f_R",
        rf"\left({tex(model.f_RR)}\right)\nabla_a\nabla_bR"
        rf"+\left({tex(model.f_RRR)}\right)"
        r"(\nabla_aR)(\nabla_bR)",
    )
    show_eq(
        r"\Box f_R",
        rf"\left({tex(model.f_RR)}\right)\Box R"
        rf"+\left({tex(model.f_RRR)}\right)(\nabla R)^2",
    )

    show_eq(
        r"-2\nabla^m\nabla^nP_{amnb}",
        derivative_term_expanded_tex(model),
        boxed=True,
    )

derive_double_divergence(model)

### 7A. Bajar los índices de \(P^{abcd}\) con el patrón requerido

<IPython.core.display.Math object>

### 7B. Aplicar dos derivadas covariantes

<IPython.core.display.Math object>

En la segunda línea se usó únicamente la compatibilidad métrica \(\nabla_\lambda g_{\mu\nu}=0\).

...

<IPython.core.display.Math object>

### 7E. Expandir la regla de la cadena covariante

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Etapa 8. Ensamblar \(E_{ab}\) usando solo las piezas ya derivadas

Las tres piezas son:

\[
\mathcal R_{ab},
\qquad
-\frac12g_{ab}L,
\qquad
-2\nabla^m\nabla^nP_{amnb}.
\]

Ahora se suman.

In [28]:
def derive_field_equation(model):
    section("8A. Forma compacta ensamblada")

    show_eq(
        r"E_{ab}",
        rf"\left({tex(model.f_R)}\right)R_{{ab}}"
        rf"-\frac12g_{{ab}}\left({tex(model.L)}\right)"
        rf"+g_{{ab}}\Box\left({tex(model.f_R)}\right)"
        rf"-\nabla_a\nabla_b\left({tex(model.f_R)}\right)",
        boxed=True,
    )

    section("8B. Forma expandida para el input actual")

    derivative_tex = derivative_term_expanded_tex(model)
    show_eq(
        r"E_{ab}",
        rf"\left({tex(model.f_R)}\right)R_{{ab}}"
        rf"-\frac12g_{{ab}}\left({tex(model.L)}\right)"
        rf"+{derivative_tex}",
        boxed=True,
    )

    section("8C. Con materia, usando la normalización del Beamer")
    show_eq(r"E_{ab}", r"\frac12T_{ab}", boxed=True)

derive_field_equation(model)

### 8A. Forma compacta ensamblada

<IPython.core.display.Math object>

### 8B. Forma expandida para el input actual

<IPython.core.display.Math object>

### 8C. Con materia, usando la normalización del Beamer

<IPython.core.display.Math object>

# Etapa 9. Diagnosticar si las ecuaciones son de segundo o cuarto orden

El Beamer termina imponiendo

\[
\nabla_aP^{abcd}=0.
\]

Para \(f(R)\), esta condición se evalúa directamente desde el \(P^{abcd}\) calculado en la etapa 2.

In [29]:
def diagnose_order(model):
    section(r"9A. Derivar covariantemente \(P^{abcd}\)")

    show_eq(
        r"\nabla_aP^{abcd}",
        r"\frac12\left["
        r"(\nabla^cf_R)g^{bd}"
        r"-(\nabla^df_R)g^{bc}"
        r"\right]",
    )
    show_eq(
        r"\nabla^cf_R",
        rf"\left({tex(model.f_RR)}\right)\nabla^cR",
    )
    show_eq(
        r"\nabla_aP^{abcd}",
        rf"\frac12\left({tex(model.f_RR)}\right)"
        r"\left[(\nabla^cR)g^{bd}"
        r"-(\nabla^dR)g^{bc}\right]",
        boxed=True,
    )

    section("9B. Diagnóstico automático")

    if sp.simplify(model.f_RR) == 0:
        note(
            r"**Resultado:** \(f_{RR}=0\). El lagrangiano es lineal en \(R\), "
            r"por lo que \(\nabla_aP^{abcd}=0\) de forma identitaria y las "
            r"ecuaciones gravitacionales son de segundo orden."
        )
    else:
        note(
            r"**Resultado:** \(f_{RR}\neq0\). En una configuración genérica, "
            r"\(\nabla_aP^{abcd}\neq0\); aparecen términos con "
            r"\(\nabla_a\nabla_bR\) y \(\Box R\). Como \(R\sim\partial^2g\), "
            r"las ecuaciones son genéricamente de cuarto orden en la métrica."
        )
        note(
            r"Una solución particular con \(R=\mathrm{const.}\) puede anular "
            r"estos términos sobre ese fondo, pero eso no convierte a la teoría "
            r"completa en una teoría de segundo orden."
        )

diagnose_order(model)

### 9A. Derivar covariantemente \(P^{abcd}\)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### 9B. Diagnóstico automático

**Resultado:** \(f_{RR}=0\). El lagrangiano es lineal en \(R\), por lo que \(\nabla_aP^{abcd}=0\) de forma identitaria y las ecuaciones gravitacionales son de segundo orden.

# Resumen automático del modelo actual

In [30]:
def compact_summary(model):
    order = (
        "Segundo orden (lineal en R)"
        if sp.simplify(model.f_RR) == 0
        else "Cuarto orden genérico"
    )

    display(Markdown(
        rf"""
- **Input:** \(L={tex(model.L)}\)
- **\(f_R\):** \({tex(model.f_R)}\)
- **\(f_{{RR}}\):** \({tex(model.f_RR)}\)
- **Diagnóstico:** **{order}**
        """
    ))

compact_summary(model)


- **Input:** \(L=R\)
- **\(f_R\):** \(1\)
- **\(f_{RR}\):** \(0\)
- **Diagnóstico:** **Segundo orden (lineal en R)**
        

# Ejemplos de validación

Estas celdas no reemplazan la deducción. Solo verifican que el motor reproduce casos límite conocidos.

In [31]:
# Caso 1: Einstein--Hilbert
EH = FRModel(R)

assert sp.simplify(EH.f_R - 1) == 0
assert sp.simplify(EH.f_RR) == 0
assert derivative_term_expanded_tex(EH) == "0"

section(r"Chequeo 1: \(L=R\)")
show_eq(r"f_R", tex(EH.f_R))
show_eq(r"-2\nabla^m\nabla^nP_{amnb}", "0")
show_eq(
    r"E_{ab}",
    r"R_{ab}-\frac12g_{ab}R",
    boxed=True,
)

### Chequeo 1: \(L=R\)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [32]:
# Caso 2: corrección cuadrática
quadratic = FRModel(R + alpha*R**2)

assert sp.simplify(quadratic.f_R - (1 + 2*alpha*R)) == 0
assert sp.simplify(quadratic.f_RR - 2*alpha) == 0
assert sp.simplify(quadratic.f_RRR) == 0

section(r"Chequeo 2: \(L=R+\alpha R^2\)")
show_eq(r"f_R", tex(quadratic.f_R))
show_eq(r"f_{RR}", tex(quadratic.f_RR))
show_eq(
    r"-2\nabla^m\nabla^nP_{amnb}",
    derivative_term_expanded_tex(quadratic),
    boxed=True,
)

### Chequeo 2: \(L=R+\alpha R^2\)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

# Función opcional: ejecutar toda la especialización \(f(R)\) para otro input

La función siguiente recorre las etapas que dependen del lagrangiano. La deducción general de Lie y Palatini no se repite porque es universal.

In [33]:
def run_fR_specialization(L_new):
    new_model = FRModel(L_new)
    new_model.summary()
    derive_P(new_model)
    derive_Rcal(new_model)
    derive_density_variation(new_model)
    derive_riemann_split(new_model)
    derive_double_divergence(new_model)
    derive_field_equation(new_model)
    diagnose_order(new_model)
    compact_summary(new_model)
    return new_model

# Ejemplo:
# otro_modelo = run_fR_specialization(R + beta*R**3)

## Lectura física del resultado

La deducción muestra algo importante:

$
P^{abcd}
= \frac{f_R}{2} (g^{ac}g^{bd}-g^{ad}g^{bc}).
$

Si $f_R$ depende de $R$, entonces $P^{abcd}$ también depende de la curvatura. Por ello,

$
\nabla\nabla P
\sim
\nabla\nabla f_R
\sim
f_{RR}\nabla\nabla R+\cdots,
$

y como

$
R\sim \partial^2g+(\partial g)^2,
$

aparecen genéricamente derivadas de cuarto orden de la métrica.

Por eso, dentro de la familia $f(R)$, la condición de Lanczos--Lovelock

$
\nabla_aP^{abcd}=0
$

selecciona de manera genérica solo los lagrangianos lineales

$
f(R)=aR+b.
$

Ese último resultado no fue introducido como hipótesis: emerge del $P^{abcd}$ calculado desde el input.